# **Prerequisites**

In [ ]:
!pip install bertopic

In [ ]:
import requests
import pandas as pd
import time
import re
from tqdm import tqdm
from collections import Counter

from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

from bertopic import BERTopic
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

In [ ]:
# ==============================
# GAME METADATA
# ==============================

def get_game_name(app_id):
    url = f"https://store.steampowered.com/api/appdetails?appids={app_id}"
    try:
        r = requests.get(url)
        data = r.json()
        if data[str(app_id)]["success"]:
            return data[str(app_id)]["data"]["name"]
    except:
        pass
    return f"Unknown Game ({app_id})"


# ==============================
# PLAYER STATS
# ==============================

def get_player_stats(app_id, game_name):
    url = f"https://api.steampowered.com/ISteamUserStats/GetNumberOfCurrentPlayers/v1/?appid={app_id}"
    try:
        r = requests.get(url)
        data = r.json()
        current_players = data["response"]["player_count"]
    except:
        current_players = "N/A"

    print(f"\n=== PLAYER SNAPSHOT: {game_name} ===")
    if current_players != "N/A":
        print(f"  Current Players:  {current_players:,}")
    else:
        print(f"  Current Players:  N/A")

# ==============================
# TEXT CLEANING
# ==============================

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-z\s]', ' ', text)
    return text


# ==============================
# DATA COLLECTION
# ==============================

def fetch_reviews(app_id, source_name, num_reviews=1000):
    url = f"https://store.steampowered.com/appreviews/{app_id}"

    params = {
        "json": 1,
        "filter": "all",
        "language": "english",
        "day_range": 3650,
        "review_type": "all",
        "purchase_type": "all",
        "num_per_page": 100
    }

    reviews = []
    cursor = "*"

    with tqdm(total=num_reviews, desc=f"Fetching {source_name}") as pbar:
        while len(reviews) < num_reviews:
            params["cursor"] = cursor

            try:
                r = requests.get(url, params=params)
                data = r.json()
            except:
                time.sleep(2)
                continue

            if not data.get("reviews"):
                break

            for review in data["reviews"]:
                author = review.get("author", {})

                reviews.append({
                    "steamid": str(author.get("steamid")),
                    "review_text": review.get("review"),
                    "recommended": review.get("voted_up"),
                    "source": source_name
                })

                pbar.update(1)

                if len(reviews) >= num_reviews:
                    break

            cursor = data.get("cursor")
            time.sleep(1)

    df = pd.DataFrame(reviews)
    df["review_text"] = df["review_text"].apply(clean_text)

    return df

In [ ]:
# ==============================
# SENTIMENT MODEL
# ==============================

def analyze_sentiment(df, label):
    X = df["review_text"].fillna("")
    y = df["recommended"].map({True: "Recommend", False: "Not Recommend"})

    vectorizer = TfidfVectorizer(max_features=5000, stop_words="english")
    X_vec = vectorizer.fit_transform(X)

    X_train, X_test, y_train, y_test = train_test_split(
        X_vec, y, test_size=0.2, random_state=42
    )

    model = LogisticRegression(max_iter=1000, class_weight="balanced")
    model.fit(X_train, y_train)

    preds = model.predict(X_test)
    print(f"\n=== SENTIMENT MODEL ({label}) ===")
    print(classification_report(y_test, preds))

    feature_names = vectorizer.get_feature_names_out()
    coef = model.coef_[0]

    n = 15
    top_negative_idx = coef.argsort()[:n]
    top_positive_idx = coef.argsort()[-n:][::-1]

    print(f"\nTop words driving NOT RECOMMEND:")
    for idx in top_negative_idx:
        print(f"  {feature_names[idx]:<25} coef: {coef[idx]:.4f}")

    print(f"\nTop words driving RECOMMEND:")
    for idx in top_positive_idx:
        print(f"  {feature_names[idx]:<25} coef: {coef[idx]:.4f}")

    return model, vectorizer


# ==============================
# BERTOPIC
# ==============================

def analyze_topics_bertopic(df):
    print("\n=== TOPIC MODEL (BERTopic) ===")

    docs = df["review_text"].dropna().tolist()

    steam_stopwords = [
        "game", "play", "played", "playing",
        "like", "just", "get", "one", "will"
    ]

    vectorizer_model = CountVectorizer(
        stop_words=list(ENGLISH_STOP_WORDS) + steam_stopwords,
        ngram_range=(1, 2),
        min_df=5
    )

    topic_model = BERTopic(
        vectorizer_model=vectorizer_model,
        verbose=True
    )

    topics, probs = topic_model.fit_transform(docs)

    topic_info = topic_model.get_topic_info()

    print("\nTop Topics Found:")
    print(topic_info.head(15))

    print("\nSample Clean Topic Words:\n")
    for topic_id in topic_info["Topic"].head(10):
        if topic_id == -1:
            continue

        words = topic_model.get_topic(topic_id)
        words = [w[0] for w in words[:10]]
        print(f"Topic {topic_id}: {', '.join(words)}")

    return topic_model


# ==============================
# TOPIC x SENTIMENT BREAKDOWN
# ==============================

def topic_sentiment_breakdown(df, topic_model, sentiment_model, vectorizer):
    docs = df["review_text"].dropna().reset_index(drop=True)
    labels = df["recommended"].dropna().reset_index(drop=True)

    topics, _ = topic_model.transform(docs.tolist())

    X_vec = vectorizer.transform(docs)
    preds = sentiment_model.predict(X_vec)

    results = pd.DataFrame({
        "topic": topics,
        "predicted_sentiment": preds,
        "actual": labels.map({True: "Recommend", False: "Not Recommend"})
    })

    rows = []
    for topic_id, group in results[results["topic"] != -1].groupby("topic"):
        total = len(group)
        neg_pct = (group["predicted_sentiment"] == "Not Recommend").mean() * 100
        pos_pct = 100 - neg_pct
        words = topic_model.get_topic(topic_id)
        label = ", ".join([w[0] for w in words[:3]])
        rows.append((topic_id, label, total, pos_pct, neg_pct))

    negative_side = sorted([r for r in rows if r[4] > 50], key=lambda x: x[4], reverse=True)
    positive_side = sorted([r for r in rows if r[4] <= 50], key=lambda x: x[3], reverse=True)

    print("\n=== TOPIC-LEVEL SENTIMENT BREAKDOWN ===")

    print("\n--- MOST NEGATIVE ---")
    for r in negative_side:
        print(f"  Topic {r[0]} [{r[1]}]")
        print(f"    Reviews: {r[2]}  |  👎 {r[4]:.1f}%  |  👍 {r[3]:.1f}%")

    print("\n--- MOST POSITIVE ---")
    for r in positive_side:
        print(f"  Topic {r[0]} [{r[1]}]")
        print(f"    Reviews: {r[2]}  |  👍 {r[3]:.1f}%  |  👎 {r[4]:.1f}%")

    return results

In [ ]:
# ==============================
# FULL PIPELINE
# ==============================

def analyze_full_game(base_app_id, dlc_app_id=None, num_reviews=4000):
    game_name = get_game_name(base_app_id)
    print(f"\n===== BASE GAME: {game_name} (ID: {base_app_id}) =====")

    # Player snapshot at the top
    get_player_stats(base_app_id, game_name)

    base_df = fetch_reviews(base_app_id, "base", num_reviews)
    model, vectorizer = analyze_sentiment(base_df, "base")
    topic_model = analyze_topics_bertopic(base_df)
    breakdown = topic_sentiment_breakdown(base_df, topic_model, model, vectorizer)

    base_positive_rate = base_df["recommended"].mean()
    print("\n=== OVERALL BASE GAME SENTIMENT ===")
    print(f"Positive Rate: {base_positive_rate:.2f}")
    print(f"Negative Rate: {1 - base_positive_rate:.2f}")

    if dlc_app_id:
        dlc_name = get_game_name(dlc_app_id)
        print(f"\n===== DLC: {dlc_name} (ID: {dlc_app_id}) =====")

        # Player snapshot for DLC
        get_player_stats(dlc_app_id, dlc_name)

        dlc_df = fetch_reviews(dlc_app_id, "dlc", num_reviews // 2)
        analyze_sentiment(dlc_df, "dlc")
        analyze_topics_bertopic(dlc_df)
        topic_sentiment_breakdown(dlc_df, topic_model, model, vectorizer)

        dlc_positive_rate = dlc_df["recommended"].mean()
        print("\n=== OVERALL DLC SENTIMENT ===")
        print(f"Positive Rate: {dlc_positive_rate:.2f}")
        print(f"Negative Rate: {1 - dlc_positive_rate:.2f}")

        print("\n=== BASE vs DLC COMPARISON ===")
        print(f"Base positive rate:  {base_positive_rate:.2f}")
        print(f"DLC positive rate:   {dlc_positive_rate:.2f}")
        diff = dlc_positive_rate - base_positive_rate
        direction = "better" if diff > 0 else "worse"
        print(f"DLC was received {abs(diff)*100:.1f}% {direction} than the base game")

    return base_df, topic_model, model, vectorizer, breakdown

# **Tests of Recent Popular Games with App ID (AID)**

In [ ]:
# ==============================================================================
  # Elden Ring            ->     AID: 1245620 (VERY Popular Game)
  # ARC Raiders           ->     AID: 1808500 (Currently Most Played Game)

  # COD Black Ops 7       ->     AID: 1938090 (NEGATIVE Reviews)
  # Shadow Of Doubt       ->     AID: 1938090 (Niche Smaller Game)

  # Ninja Gaiden 4        ->     AID: 2627260
  #  ''     ''   '' DLC   ->     AID: 4191490 (Test with DLC Added)

  # Marvel Rivals         ->     AID: 2767030 (Popular Live Service Game)
  # HellDivers 2          ->     AID: 553850  (Live Service Game)
# ==============================================================================


# ==============================
# BEGIN TESTS
# ==============================

# Elden Ring
data = analyze_full_game(1245620)


===== BASE GAME: ELDEN RING (ID: 1245620) =====

=== PLAYER SNAPSHOT: ELDEN RING ===
  Current Players:  25,720


Fetching base: 100%|██████████| 4000/4000 [00:56<00:00, 70.53it/s]
2026-05-06 22:37:06,909 - BERTopic - Embedding - Transforming documents to embeddings.



=== SENTIMENT MODEL (base) ===
               precision    recall  f1-score   support

Not Recommend       0.36      0.33      0.35        24
    Recommend       0.98      0.98      0.98       776

     accuracy                           0.96       800
    macro avg       0.67      0.66      0.66       800
 weighted avg       0.96      0.96      0.96       800


Top words driving NOT RECOMMEND:
  terrible                  coef: -3.9396
  save                      coef: -2.9179
  files                     coef: -2.8080
  insan                     coef: -2.7987
  fans                      coef: -2.7735
  completely                coef: -2.6401
  unplayable                coef: -2.6220
  fix                       coef: -2.6179
  tried                     coef: -2.5153
  don                       coef: -2.4664
  anymore                   coef: -2.4404
  taken                     coef: -2.3561
  garbage                   coef: -2.3507
  slow                      coef: -2.3142
  doesn      

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

2026-05-06 22:41:00,217 - BERTopic - Embedding - Completed ✓
2026-05-06 22:41:00,218 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-05-06 22:41:37,681 - BERTopic - Dimensionality - Completed ✓
2026-05-06 22:41:37,683 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-05-06 22:41:37,877 - BERTopic - Cluster - Completed ✓
2026-05-06 22:41:37,885 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-05-06 22:41:38,518 - BERTopic - Representation - Completed ✓



Top Topics Found:
    Topic  Count                                 Name  \
0      -1   1364              -1_world_games_time_fun   
1       0    787        0_elden_ring_elden ring_world   
2       1    231               1_bosses_boss_beat_fun   
3       2    183       2_souls_souls games_games_best   
4       3    108     3_fun_hard_difficult_challenging   
5       4     88                   4_ps_pc_xbox_hours   
6       5     77               5_seamless_coop_mod_op   
7       6     76                6_fps_pc_runs_support   
8       7     72            7_souls_boss_bosses_fight   
9       8     70   8_fromsoft_fromsoftware_games_best   
10      9     63                   9_dlc_buy_ng_worth   
11     10     59   10_open world_open_world_best open   
12     11     51        11_hate_rage_recommend_stress   
13     12     48  12_dark souls_dark_souls_bloodborne   
14     13     41         13_lore_story_combat_amazing   

                                       Representation  \
0   [world,

Batches:   0%|          | 0/125 [00:00<?, ?it/s]

2026-05-06 22:45:30,789 - BERTopic - Dimensionality - Reducing dimensionality of input embeddings.
2026-05-06 22:45:30,809 - BERTopic - Dimensionality - Completed ✓
2026-05-06 22:45:30,810 - BERTopic - Clustering - Approximating new points with `hdbscan_model`
2026-05-06 22:45:30,972 - BERTopic - Cluster - Completed ✓



=== TOPIC-LEVEL SENTIMENT BREAKDOWN ===

--- MOST NEGATIVE ---

--- MOST POSITIVE ---
  Topic 3 [fun, hard, difficult]
    Reviews: 108  |  👍 100.0%  |  👎 0.0%
  Topic 10 [open world, open, world]
    Reviews: 59  |  👍 100.0%  |  👎 0.0%
  Topic 11 [hate, rage, recommend]
    Reviews: 51  |  👍 100.0%  |  👎 0.0%
  Topic 12 [dark souls, dark, souls]
    Reviews: 48  |  👍 100.0%  |  👎 0.0%
  Topic 13 [lore, story, combat]
    Reviews: 41  |  👍 100.0%  |  👎 0.0%
  Topic 15 [story, amazing, graphics]
    Reviews: 39  |  👍 100.0%  |  👎 0.0%
  Topic 16 [miyazaki, thank, best]
    Reviews: 34  |  👍 100.0%  |  👎 0.0%
  Topic 17 [malenia, sex, rot]
    Reviews: 33  |  👍 100.0%  |  👎 0.0%
  Topic 19 [replay, time, best time]
    Reviews: 31  |  👍 100.0%  |  👎 0.0%
  Topic 20 [dlc, base, new]
    Reviews: 30  |  👍 100.0%  |  👎 0.0%
  Topic 22 [peak, games peak, beautiful visuals]
    Reviews: 30  |  👍 100.0%  |  👎 0.0%
  Topic 23 [greatest, best, said]
    Reviews: 28  |  👍 100.0%  |  👎 0.0%
  Top

In [ ]:
# ARC Raiders
data = analyze_full_game(1808500)


===== BASE GAME: ARC Raiders (ID: 1808500) =====

=== PLAYER SNAPSHOT: ARC Raiders ===
  Current Players:  69,887


Fetching base: 100%|██████████| 4000/4000 [00:58<00:00, 68.12it/s]
2026-05-06 22:46:30,976 - BERTopic - Embedding - Transforming documents to embeddings.



=== SENTIMENT MODEL (base) ===
               precision    recall  f1-score   support

Not Recommend       0.86      0.90      0.88       414
    Recommend       0.88      0.84      0.86       386

     accuracy                           0.87       800
    macro avg       0.87      0.87      0.87       800
 weighted avg       0.87      0.87      0.87       800


Top words driving NOT RECOMMEND:
  just                      coef: -3.6628
  boring                    coef: -3.0100
  cheaters                  coef: -2.8252
  pvp                       coef: -2.6319
  toxic                     coef: -2.5423
  ruined                    coef: -2.2575
  players                   coef: -2.1385
  worst                     coef: -2.0927
  trash                     coef: -2.0846
  killed                    coef: -2.0836
  worse                     coef: -1.8099
  banned                    coef: -1.7541
  update                    coef: -1.7460
  camping                   coef: -1.6520
  rats       

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

2026-05-06 22:51:37,197 - BERTopic - Embedding - Completed ✓
2026-05-06 22:51:37,198 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-05-06 22:52:14,029 - BERTopic - Dimensionality - Completed ✓
2026-05-06 22:52:14,031 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-05-06 22:52:14,361 - BERTopic - Cluster - Completed ✓
2026-05-06 22:52:14,373 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-05-06 22:52:15,775 - BERTopic - Representation - Completed ✓



Top Topics Found:
    Topic  Count                                              Name  \
0      -1   1488                         -1_pvp_people_players_pve   
1       0    704                 0_raiders_arc_arc raiders_players   
2       1    204  1_extraction_shooter_extraction shooter_shooters   
3       2    200                                2_pvp_pve_fun_mode   
4       3    153       3_extraction_extraction shooter_shooter_pvp   
5       4     98                                4_rats_rat_pve_pvp   
6       5     89             5_tarkov_shooter_escape tarkov_escape   
7       6     86                          6_fun_friends_solo_great   
8       7     79                           7_toxic_people_kill_fun   
9       8     67                         8_solo_friendly_solos_duo   
10      9     64                    9_cheaters_devs_hackers_cheats   
11     10     62                       10_banned_support_ban_cheat   
12     11     50                    11_embark_update_make_terrible   
1

Batches:   0%|          | 0/125 [00:00<?, ?it/s]

2026-05-06 22:57:22,589 - BERTopic - Dimensionality - Reducing dimensionality of input embeddings.
2026-05-06 22:57:22,608 - BERTopic - Dimensionality - Completed ✓
2026-05-06 22:57:22,609 - BERTopic - Clustering - Approximating new points with `hdbscan_model`
2026-05-06 22:57:22,769 - BERTopic - Cluster - Completed ✓



=== TOPIC-LEVEL SENTIMENT BREAKDOWN ===

--- MOST NEGATIVE ---
  Topic 15 [expedition, million, mil]
    Reviews: 37  |  👎 100.0%  |  👍 0.0%
  Topic 17 [guns, fun, free]
    Reviews: 35  |  👎 100.0%  |  👍 0.0%
  Topic 24 [players, fix, changes]
    Reviews: 26  |  👎 100.0%  |  👍 0.0%
  Topic 26 [pvp, players, pve]
    Reviews: 23  |  👎 100.0%  |  👍 0.0%
  Topic 31 [cheaters, cheating, embark]
    Reviews: 19  |  👎 100.0%  |  👍 0.0%
  Topic 9 [cheaters, devs, hackers]
    Reviews: 64  |  👎 98.4%  |  👍 1.6%
  Topic 10 [banned, support, ban]
    Reviews: 62  |  👎 98.4%  |  👍 1.6%
  Topic 14 [pvp, players, shoot]
    Reviews: 41  |  👎 97.6%  |  👍 2.4%
  Topic 28 [crashes, restart, error]
    Reviews: 21  |  👎 95.2%  |  👍 4.8%
  Topic 13 [durability, nerf, weapons]
    Reviews: 44  |  👎 93.2%  |  👍 6.8%
  Topic 4 [rats, rat, pve]
    Reviews: 98  |  👎 92.9%  |  👍 7.1%
  Topic 20 [pvp, pve, players]
    Reviews: 31  |  👎 90.3%  |  👍 9.7%
  Topic 22 [late, spawn, free]
    Reviews: 27  |  👎 

In [ ]:
# COD Black Ops 7 (REVIEWED BOMBED)
data = analyze_full_game(1938090)


===== BASE GAME: Call of Duty® (ID: 1938090) =====

=== PLAYER SNAPSHOT: Call of Duty® ===
  Current Players:  30,945


Fetching base: 100%|██████████| 4000/4000 [00:49<00:00, 80.60it/s]
2026-05-06 22:58:13,842 - BERTopic - Embedding - Transforming documents to embeddings.



=== SENTIMENT MODEL (base) ===
               precision    recall  f1-score   support

Not Recommend       0.94      0.90      0.92       665
    Recommend       0.60      0.70      0.65       135

     accuracy                           0.87       800
    macro avg       0.77      0.80      0.78       800
 weighted avg       0.88      0.87      0.87       800


Top words driving NOT RECOMMEND:
  worst                     coef: -2.4766
  just                      coef: -2.4711
  terrible                  coef: -2.2267
  trash                     coef: -2.1342
  garbage                   coef: -2.1199
  time                      coef: -2.1089
  update                    coef: -2.0229
  waste                     coef: -1.9888
  anymore                   coef: -1.9609
  money                     coef: -1.8736
  franchise                 coef: -1.8551
  activision                coef: -1.8210
  horrible                  coef: -1.7471
  making                    coef: -1.6809
  worse      

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

2026-05-06 23:01:19,140 - BERTopic - Embedding - Completed ✓
2026-05-06 23:01:19,142 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-05-06 23:01:56,387 - BERTopic - Dimensionality - Completed ✓
2026-05-06 23:01:56,389 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-05-06 23:01:56,569 - BERTopic - Cluster - Completed ✓
2026-05-06 23:01:56,576 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-05-06 23:01:57,039 - BERTopic - Representation - Completed ✓



Top Topics Found:
    Topic  Count                                               Name  \
0      -1   1582                         -1_black_cod_ops_black ops   
1       0    264                      0_gb_download_install_warzone   
2       1    199                        1_cod_worst cod_worst_games   
3       2    188                          2_fun_good_recommend_love   
4       3    110            3_warzone_royale_blackout_battle royale   
5       4    110                        4_trash_money_free_terrible   
6       5    103                        5_duty_year_franchise_money   
7       6    103                   6_cheaters_cheat_anti cheat_anti   
8       7     83                  7_zombies_multiplayer_mode_zombie   
9       8     72                      8_tpm_boot_secure_secure boot   
10      9     67                             9_bo_bo bo_better_beta   
11     10     67                       10_banned_ban_shadow_account   
12     11     60                 11_ops_black ops_black_mu

Batches:   0%|          | 0/125 [00:00<?, ?it/s]

2026-05-06 23:05:01,140 - BERTopic - Dimensionality - Reducing dimensionality of input embeddings.
2026-05-06 23:05:01,158 - BERTopic - Dimensionality - Completed ✓
2026-05-06 23:05:01,160 - BERTopic - Clustering - Approximating new points with `hdbscan_model`
2026-05-06 23:05:01,308 - BERTopic - Cluster - Completed ✓



=== TOPIC-LEVEL SENTIMENT BREAKDOWN ===

--- MOST NEGATIVE ---
  Topic 13 [restart, launch, closes]
    Reviews: 54  |  👎 100.0%  |  👍 0.0%
  Topic 22 [account, phone number, phone]
    Reviews: 33  |  👎 100.0%  |  👍 0.0%
  Topic 24 [activision, company, cuz]
    Reviews: 32  |  👎 100.0%  |  👍 0.0%
  Topic 32 [refund, steam, hours]
    Reviews: 24  |  👎 100.0%  |  👍 0.0%
  Topic 33 [shaders, restart, update]
    Reviews: 22  |  👎 100.0%  |  👍 0.0%
  Topic 38 [launcher, retarded, worst]
    Reviews: 17  |  👎 100.0%  |  👍 0.0%
  Topic 41 [activision, cheaters, care]
    Reviews: 15  |  👎 100.0%  |  👍 0.0%
  Topic 43 [slop, ai slop, ai]
    Reviews: 15  |  👎 100.0%  |  👍 0.0%
  Topic 46 [warzone, ui, broken]
    Reviews: 14  |  👎 100.0%  |  👍 0.0%
  Topic 48 [skins, dollars, players]
    Reviews: 10  |  👎 100.0%  |  👍 0.0%
  Topic 8 [tpm, boot, secure]
    Reviews: 72  |  👎 98.6%  |  👍 1.4%
  Topic 0 [gb, download, install]
    Reviews: 264  |  👎 98.1%  |  👍 1.9%
  Topic 28 [account, log

In [ ]:
# Shadow Of Doubt
data = analyze_full_game(986130)


===== BASE GAME: Shadows of Doubt (ID: 986130) =====

=== PLAYER SNAPSHOT: Shadows of Doubt ===
  Current Players:  202


Fetching base: 100%|██████████| 4000/4000 [00:53<00:00, 74.55it/s]
2026-05-06 23:05:55,943 - BERTopic - Embedding - Transforming documents to embeddings.



=== SENTIMENT MODEL (base) ===
               precision    recall  f1-score   support

Not Recommend       0.84      0.92      0.88       252
    Recommend       0.96      0.92      0.94       548

     accuracy                           0.92       800
    macro avg       0.90      0.92      0.91       800
 weighted avg       0.92      0.92      0.92       800


Top words driving NOT RECOMMEND:
  idea                      coef: -2.6693
  unfinished                coef: -2.6535
  just                      coef: -2.4831
  minutes                   coef: -2.4655
  boring                    coef: -2.3917
  state                     coef: -2.3585
  performance               coef: -2.3454
  early                     coef: -2.2868
  buggy                     coef: -2.2567
  concept                   coef: -2.2479
  fix                       coef: -2.2461
  access                    coef: -2.2144
  finished                  coef: -2.2032
  release                   coef: -2.1707
  bad        

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

2026-05-06 23:09:24,520 - BERTopic - Embedding - Completed ✓
2026-05-06 23:09:24,522 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-05-06 23:10:01,597 - BERTopic - Dimensionality - Completed ✓
2026-05-06 23:10:01,600 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-05-06 23:10:01,784 - BERTopic - Cluster - Completed ✓
2026-05-06 23:10:01,792 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-05-06 23:10:02,406 - BERTopic - Representation - Completed ✓



Top Topics Found:
    Topic  Count                                    Name  \
0      -1    353        -1_steam deck_case_building_step   
1       0     66  0_fun_games_definitely recommend_enjoy   
2       1     56            1_fun_challenging_worth_hard   
3       2     56  2_community_modifiers_unfinished_early   
4       3     51       3_rat detective_detective_rat_man   
5       4     48             4_ask_doing_gov_environment   
6       5     48       5_buggy mess_buggy_properly_awful   
7       6     46     6_love detective_exactly_poor_space   
8       7     46            7_turned_tired_solved_robbed   
9       8     45      8_player_simulation_doubt_murderer   
10      9     45           9_buggy_reload_state_tutorial   
11     10     44                                  10____   
12     11     43           11_overall_easier_glitchy_bad   
13     12     42           12_batman_greatly_hobo_secret   
14     13     41      13_fps_dlss_optimized_optimization   

                    

Batches:   0%|          | 0/125 [00:00<?, ?it/s]

2026-05-06 23:13:33,019 - BERTopic - Dimensionality - Reducing dimensionality of input embeddings.
2026-05-06 23:13:33,038 - BERTopic - Dimensionality - Completed ✓
2026-05-06 23:13:33,039 - BERTopic - Clustering - Approximating new points with `hdbscan_model`
2026-05-06 23:13:33,180 - BERTopic - Cluster - Completed ✓



=== TOPIC-LEVEL SENTIMENT BREAKDOWN ===

--- MOST NEGATIVE ---
  Topic 38 [execution, idea, demo]
    Reviews: 30  |  👎 90.0%  |  👍 10.0%
  Topic 54 [npc, talk, tedious]
    Reviews: 25  |  👎 88.0%  |  👍 12.0%
  Topic 64 [unplayable, wasted, waste]
    Reviews: 23  |  👎 87.0%  |  👍 13.0%
  Topic 73 [auto, basic, alot]
    Reviews: 21  |  👎 85.7%  |  👍 14.3%
  Topic 127 [returned, runs, great concept]
    Reviews: 14  |  👎 85.7%  |  👍 14.3%
  Topic 89 [crashes, memory, available]
    Reviews: 18  |  👎 83.3%  |  👍 16.7%
  Topic 152 [horrible, load, trouble]
    Reviews: 11  |  👎 81.8%  |  👍 18.2%
  Topic 109 [height, target, useful]
    Reviews: 15  |  👎 80.0%  |  👍 20.0%
  Topic 113 [keys, stop working, actions]
    Reviews: 15  |  👎 80.0%  |  👍 20.0%
  Topic 121 [unconscious, cares, npcs]
    Reviews: 14  |  👎 78.6%  |  👍 21.4%
  Topic 33 [tutorial, slow, procedural generation]
    Reviews: 31  |  👎 77.4%  |  👍 22.6%
  Topic 2 [community, modifiers, unfinished]
    Reviews: 56  |  👎 7

In [ ]:
# Ninja Gaiden 4
# data = analyze_full_game(2627260)

# Ninja Gaiden 4 w/ DLC
data = analyze_full_game(2627260, dlc_app_id=4191490)


===== BASE GAME: NINJA GAIDEN 4 (ID: 2627260) =====

=== PLAYER SNAPSHOT: NINJA GAIDEN 4 ===
  Current Players:  106


Fetching base: 100%|██████████| 4000/4000 [00:52<00:00, 76.07it/s]
2026-05-06 23:14:26,891 - BERTopic - Embedding - Transforming documents to embeddings.



=== SENTIMENT MODEL (base) ===
               precision    recall  f1-score   support

Not Recommend       0.67      0.91      0.77       108
    Recommend       0.98      0.93      0.96       692

     accuracy                           0.93       800
    macro avg       0.83      0.92      0.86       800
 weighted avg       0.94      0.93      0.93       800


Top words driving NOT RECOMMEND:
  boring                    coef: -3.3491
  just                      coef: -2.9441
  bad                       coef: -2.8810
  refund                    coef: -2.6354
  refunded                  coef: -2.2368
  sucks                     coef: -2.1831
  change                    coef: -2.1453
  worse                     coef: -2.1320
  trash                     coef: -2.1261
  sale                      coef: -2.0418
  paint                     coef: -2.0087
  broken                    coef: -2.0051
  terrible                  coef: -1.9934
  crash                     coef: -1.9607
  hayabusa   

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

2026-05-06 23:24:17,705 - BERTopic - Embedding - Completed ✓
2026-05-06 23:24:17,708 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-05-06 23:24:33,712 - BERTopic - Dimensionality - Completed ✓
2026-05-06 23:24:33,714 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-05-06 23:24:33,920 - BERTopic - Cluster - Completed ✓
2026-05-06 23:24:33,927 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-05-06 23:24:34,345 - BERTopic - Representation - Completed ✓



Top Topics Found:
    Topic  Count                                     Name  \
0       0     19  0_really good_disappointing_level_price   
1       1     19  1_playable_campaign_disappointing_ayane   
2       2     18                     2_going_ng_far_doesn   
3       3     18               3_maybe_budget_levels_didn   
4       4     18       4_honestly_scythe_yakumo_basically   
5       5     18                5_compared_big_people_lot   
6       6     18               6_cut_cut content_buy_make   
7       7     18               7_cool_sale_character_main   
8       8     18                 8_half_way_new enemy_cut   
9       9     18                          9_say_fun_boss_   
10     10     18  10_experience_weapons ryu_attacks_quite   
11     11     18                 11_people_pretty_way_fun   
12     12     18              12_single_ve_literally_main   
13     13     18           13_chapter_ll_extremely_review   
14     14     18         14_love_hope_ninja gaiden_gaiden   

    

Batches:   0%|          | 0/63 [00:00<?, ?it/s]

2026-05-06 23:26:19,530 - BERTopic - Dimensionality - Reducing dimensionality of input embeddings.
2026-05-06 23:26:55,903 - BERTopic - Dimensionality - Completed ✓
2026-05-06 23:26:55,906 - BERTopic - Clustering - Approximating new points with `hdbscan_model`
2026-05-06 23:26:56,030 - BERTopic - Cluster - Completed ✓



=== TOPIC-LEVEL SENTIMENT BREAKDOWN ===

--- MOST NEGATIVE ---
  Topic 9 [series, ryu, new]
    Reviews: 36  |  👎 100.0%  |  👍 0.0%
  Topic 25 [dead alive, team, alive]
    Reviews: 18  |  👎 100.0%  |  👍 0.0%
  Topic 28 [ng, enemies, ng black]
    Reviews: 18  |  👎 100.0%  |  👍 0.0%
  Topic 38 [ng, ease, releases]
    Reviews: 18  |  👎 100.0%  |  👍 0.0%
  Topic 40 [peak, lord, peak combat]
    Reviews: 18  |  👎 100.0%  |  👍 0.0%
  Topic 79 [shorter, yakumo, ryu]
    Reviews: 54  |  👎 100.0%  |  👍 0.0%
  Topic 99 [speak, voice, english]
    Reviews: 18  |  👎 100.0%  |  👍 0.0%
  Topic 88 [ryu, generic, weapons]
    Reviews: 73  |  👎 75.3%  |  👍 24.7%

--- MOST POSITIVE ---
  Topic 0 [ps, lol, came]
    Reviews: 36  |  👍 100.0%  |  👎 0.0%
  Topic 3 [amazing, really good, good]
    Reviews: 18  |  👍 100.0%  |  👎 0.0%
  Topic 4 [ryu, kinda, combat great]
    Reviews: 32  |  👍 100.0%  |  👎 0.0%
  Topic 6 [peak, absolute peak, holy]
    Reviews: 18  |  👍 100.0%  |  👎 0.0%
  Topic 21 [combat,

In [ ]:
# Marvel Rivals
data = analyze_full_game(2767030)


===== BASE GAME: Marvel Rivals (ID: 2767030) =====

=== PLAYER SNAPSHOT: Marvel Rivals ===
  Current Players:  74,886


Fetching base: 100%|██████████| 4000/4000 [00:56<00:00, 70.91it/s]
2026-05-07 00:09:43,239 - BERTopic - Embedding - Transforming documents to embeddings.



=== SENTIMENT MODEL (base) ===
               precision    recall  f1-score   support

Not Recommend       0.93      0.86      0.90       605
    Recommend       0.65      0.81      0.72       195

     accuracy                           0.85       800
    macro avg       0.79      0.84      0.81       800
 weighted avg       0.87      0.85      0.85       800


Top words driving NOT RECOMMEND:
  eomm                      coef: -2.8832
  matchmaking               coef: -2.6019
  worst                     coef: -2.5865
  worse                     coef: -2.5426
  making                    coef: -2.2915
  terrible                  coef: -2.2054
  anymore                   coef: -2.1349
  garbage                   coef: -2.1343
  just                      coef: -2.0924
  season                    coef: -1.9815
  trash                     coef: -1.9214
  bots                      coef: -1.8854
  unplayable                coef: -1.8016
  fix                       coef: -1.7999
  don        

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

2026-05-07 00:13:33,510 - BERTopic - Embedding - Completed ✓
2026-05-07 00:13:33,512 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-05-07 00:14:21,386 - BERTopic - Dimensionality - Completed ✓
2026-05-07 00:14:21,389 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-05-07 00:14:21,583 - BERTopic - Cluster - Completed ✓
2026-05-07 00:14:21,590 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-05-07 00:14:22,391 - BERTopic - Representation - Completed ✓



Top Topics Found:
    Topic  Count                                     Name  \
0      -1   1719           -1_fun_matchmaking_team_people   
1       0    290            0_eomm_matchmaking_win_ranked   
2       1    138              1_fun_shooter_es_characters   
3       2    119   2_toxic_community_chat_toxic community   
4       3    119           3_bots_bot_matches_bot matches   
5       4    116       4_marvel_marvel rivals_rivals_hero   
6       5     94  5_overwatch_better overwatch_better_fun   
7       6     89           6_marvel_love_characters_great   
8       7     81      7_balancing_characters_balance_buff   
9       8     75  8_matchmaking_good_good matchmaking_bad   
10      9     71      9_hate_life_addiction_mental health   
11     10     68       10_crashes_crashing_crash_computer   
12     11     64          11_matchmaking_win_team_matches   
13     12     57      12_marvel rivals_rivals_marvel_team   
14     13     53             13_dps_healers_healer_damage   

    

Batches:   0%|          | 0/125 [00:00<?, ?it/s]

2026-05-07 00:18:22,433 - BERTopic - Dimensionality - Reducing dimensionality of input embeddings.
2026-05-07 00:18:22,453 - BERTopic - Dimensionality - Completed ✓
2026-05-07 00:18:22,455 - BERTopic - Clustering - Approximating new points with `hdbscan_model`
2026-05-07 00:18:22,612 - BERTopic - Cluster - Completed ✓



=== TOPIC-LEVEL SENTIMENT BREAKDOWN ===

--- MOST NEGATIVE ---
  Topic 23 [balancing, unbalanced, matchmaking]
    Reviews: 27  |  👎 100.0%  |  👍 0.0%
  Topic 24 [engagement, engagement based, based matchmaking]
    Reviews: 26  |  👎 100.0%  |  👍 0.0%
  Topic 25 [comp, comp matchmaking, horrible]
    Reviews: 24  |  👎 100.0%  |  👍 0.0%
  Topic 27 [ranked, matchmaking, dont]
    Reviews: 22  |  👎 100.0%  |  👍 0.0%
  Topic 30 [steamrolled, matchmaking, skill]
    Reviews: 21  |  👎 100.0%  |  👍 0.0%
  Topic 38 [grandmaster, rank, gold]
    Reviews: 16  |  👎 100.0%  |  👍 0.0%
  Topic 40 [matchmaking, fix matchmaking, fix]
    Reviews: 14  |  👎 100.0%  |  👍 0.0%
  Topic 42 [people, dps, players]
    Reviews: 13  |  👎 100.0%  |  👍 0.0%
  Topic 50 [money, skins, making]
    Reviews: 11  |  👎 100.0%  |  👍 0.0%
  Topic 52 [rigged, enemies, streak]
    Reviews: 10  |  👎 100.0%  |  👍 0.0%
  Topic 11 [matchmaking, win, team]
    Reviews: 64  |  👎 98.4%  |  👍 1.6%
  Topic 3 [bots, bot, matches]
  

In [ ]:
# HellDivers 2
data = analyze_full_game(553850)


===== BASE GAME: HELLDIVERS™ 2 (ID: 553850) =====

=== PLAYER SNAPSHOT: HELLDIVERS™ 2 ===
  Current Players:  32,218


Fetching base: 100%|██████████| 4000/4000 [00:58<00:00, 68.68it/s]
2026-05-07 00:19:22,288 - BERTopic - Embedding - Transforming documents to embeddings.



=== SENTIMENT MODEL (base) ===
               precision    recall  f1-score   support

Not Recommend       0.97      0.96      0.96       692
    Recommend       0.74      0.80      0.77       108

     accuracy                           0.94       800
    macro avg       0.85      0.88      0.87       800
 weighted avg       0.94      0.94      0.94       800


Top words driving NOT RECOMMEND:
  devs                      coef: -4.0840
  fix                       coef: -3.2619
  arrowhead                 coef: -3.2225
  game                      coef: -2.3831
  balance                   coef: -2.3702
  performance               coef: -2.3002
  balancing                 coef: -2.2594
  unplayable                coef: -2.1192
  warbond                   coef: -2.1134
  warbonds                  coef: -2.0894
  issues                    coef: -2.0533
  pc                        coef: -2.0241
  anymore                   coef: -2.0128
  crashes                   coef: -1.9766
  community  

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

2026-05-07 00:24:36,525 - BERTopic - Embedding - Completed ✓
2026-05-07 00:24:36,526 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-05-07 00:25:12,985 - BERTopic - Dimensionality - Completed ✓
2026-05-07 00:25:12,988 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-05-07 00:25:13,185 - BERTopic - Cluster - Completed ✓
2026-05-07 00:25:13,192 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-05-07 00:25:14,531 - BERTopic - Representation - Completed ✓



Top Topics Found:
    Topic  Count                                      Name  \
0      -1   1683                 -1_devs_new_fun_community   
1       0    491            0_arrowhead_community_fun_time   
2       1    232             1_helldivers_fun_time_players   
3       2    180          2_pc_crashes_crashing_unplayable   
4       3    137      3_community_doxxed_charity_challenge   
5       4    110              4_fun_friends_recommend_best   
6       5     94  5_arrowhead_helldivers_players_community   
7       6     92    6_credits_super credits_super_warbonds   
8       7     92               7_devs_nerf_enemies_players   
9       8     83     8_balance_balancing_balance team_team   
10      9     77                9_warbonds_new_warbond_fun   
11     10     68             10_performance_pc_fps_crashes   
12     11     65          11_devs_hate_developers_fun devs   
13     12     51                12_mechs_mech_warbond_buff   
14     13     50            13_weapons_enemies_weap

Batches:   0%|          | 0/125 [00:00<?, ?it/s]

2026-05-07 00:30:22,375 - BERTopic - Dimensionality - Reducing dimensionality of input embeddings.
2026-05-07 00:30:22,394 - BERTopic - Dimensionality - Completed ✓
2026-05-07 00:30:22,395 - BERTopic - Clustering - Approximating new points with `hdbscan_model`
2026-05-07 00:30:22,545 - BERTopic - Cluster - Completed ✓



=== TOPIC-LEVEL SENTIMENT BREAKDOWN ===

--- MOST NEGATIVE ---
  Topic 12 [mechs, mech, warbond]
    Reviews: 51  |  👎 100.0%  |  👍 0.0%
  Topic 17 [settings, stutters, stuttering]
    Reviews: 31  |  👎 100.0%  |  👍 0.0%
  Topic 18 [warbond, warbonds, arrowhead]
    Reviews: 27  |  👎 100.0%  |  👍 0.0%
  Topic 24 [devs, listen, community]
    Reviews: 19  |  👎 100.0%  |  👍 0.0%
  Topic 26 [enemies, vox, kill]
    Reviews: 17  |  👎 100.0%  |  👍 0.0%
  Topic 28 [content, pay, free]
    Reviews: 16  |  👎 100.0%  |  👍 0.0%
  Topic 31 [gameguard, anti cheat, cheat]
    Reviews: 15  |  👎 100.0%  |  👍 0.0%
  Topic 33 [performance, patch, bugs]
    Reviews: 14  |  👎 100.0%  |  👍 0.0%
  Topic 35 [risk, hardware, pc]
    Reviews: 12  |  👎 100.0%  |  👍 0.0%
  Topic 37 [coyote, nerf, patch]
    Reviews: 11  |  👎 100.0%  |  👍 0.0%
  Topic 38 [devs, bugs, balance]
    Reviews: 10  |  👎 100.0%  |  👍 0.0%
  Topic 3 [community, doxxed, charity]
    Reviews: 137  |  👎 99.3%  |  👍 0.7%
  Topic 9 [warbond